In [1]:
# Imports
import os
from dotenv import load_dotenv

import numpy as np
from tqdm.notebook import tqdm

from huggingface_hub import login

import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


from pricer.items import Item
from pricer.evaluator import evaluate

from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.model_selection import train_test_split

/Users/seunaminu/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
LITE_MODE = True
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 20,000 training items, 1,000 validation items, 1,000 test items


In [3]:
LITE_MODE = True

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [4]:
# class NeuralNetwork(nn.Module):
#     def __init__(self, input_size):
#         super(NeuralNetwork, self).__init__()
#         self.layer1 = nn.Linear(input_size, 128)
#         self.layer2 = nn.Linear(128, 64)
#         self.layer3 = nn.Linear(64, 64)
#         self.layer4 = nn.Linear(64, 64)
#         self.layer5 = nn.Linear(64, 64)
#         self.layer6 = nn.Linear(64, 64)
#         self.layer7 = nn.Linear(64, 64)
#         self.layer8 = nn.Linear(64, 1)
#         self.relu = nn.ReLU()

#     def forward(self, x):
#         output1 = self.relu(self.layer1(x))
#         output2 = self.relu(self.layer2(output1))
#         output3 = self.relu(self.layer3(output2))
#         output4 = self.relu(self.layer4(output3))
#         output5 = self.relu(self.layer5(output4))
#         output6 = self.relu(self.layer6(output5))
#         output7 = self.relu(self.layer7(output6))
#         output8 = self.layer8(output7)
#         return output8

In [5]:
class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.model(x)

In [6]:
y = np.array([float(item.price) for item in train])

documents = [item.summary for item in train]

In [7]:
np.random.seed(42)
vectorizer = HashingVectorizer(n_features=5000, stop_words='english', binary=True)
X = vectorizer.fit_transform(documents)

In [8]:
# Convert data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X.toarray())
y_train_tensor = torch.FloatTensor(y).unsqueeze(1)

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train_tensor, y_train_tensor, test_size=0.1, random_state=42)

# Create the loader
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Initialize the model
input_size = X_train_tensor.shape[1]
model = NeuralNetwork(input_size)

In [9]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Number of trainable parameters: {trainable_params:,}")

Number of trainable parameters: 1,321,473


In [19]:
loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 4

for epoch in range(EPOCHS):
    model.train()
    for batch_X, batch_y in tqdm(train_loader):
        optimizer.zero_grad()

        # The next 4 lines are the 4 stages of training: forward pass, loss calculation, backward pass, optimize
        
        outputs = model(batch_X)
        loss = loss_function(outputs, batch_y)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = loss_function(val_outputs, y_val)

    print(f'Epoch [{epoch+1}/{EPOCHS}], Train Loss: {loss.item():.3f}, Val Loss: {val_loss.item():.3f}')

  0%|          | 0/282 [00:00<?, ?it/s]

Epoch [1/4], Train Loss: 458.354, Val Loss: 19490.594


  0%|          | 0/282 [00:00<?, ?it/s]

Epoch [2/4], Train Loss: 1663.851, Val Loss: 19512.236


  0%|          | 0/282 [00:00<?, ?it/s]

Epoch [3/4], Train Loss: 3250.194, Val Loss: 19386.514


  0%|          | 0/282 [00:00<?, ?it/s]

Epoch [4/4], Train Loss: 476.675, Val Loss: 19183.678


In [21]:
def neural_network(item):
    model.eval()
    with torch.no_grad():
        vector = vectorizer.transform([item.summary])
        vector = torch.FloatTensor(vector.toarray())
        result = model(vector)[0].item()
    return max(0, result)

In [22]:
evaluate(neural_network, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$135 $26 $35 $1 $14 $138 $51 $51 $17 $175 $339 $156 $44 $249 $82 $1 $25 $17 $95 $202 $33 $44 $138 $94 $261 $300 $313 $22 $40 $32 $40 $72 $85 $3 $119 $210 $4 $101 $139 $44 $166 $42 $24 $42 $108 $31 $40 $128 $9 $13 $5 $41 $154 $77 $112 $58 $19 $89 $68 $22 $23 $0 $5 $26 $425 $13 $42 $225 $6 $220 $17 $13 $70 $52 $7 $42 $20 $49 $21 $12 $85 $79 $1 $49 $6 $19 $87 $48 $115 $120 $46 $29 $11 $15 $34 $138 $48 $65 $99 $224 $28 $28 $5 $60 $59 $61 $36 $298 $4 $116 $33 $53 $110 $29 $12 $152 $147 $79 $63 $16 $22 $327 $7 $52 $76 $5 $1 $110 $1 $50 $22 $23 $87 $0 $135 $8 $61 $66 $57 $156 $35 $105 $3 $266 $191 $33 $29 $323 $87 $14 $1 $121 $11 $16 $12 $97 $94 $12 $59 $19 $107 $10 $19 $36 $445 $1 $91 $35 $51 $10 $7 $8 $200 $5 $23 $26 $2 $6 $50 $13 $340 $4 $103 $20 $17 $19 $28 $6 $7 $3 $25 $37 $37 $57 $16 $17 $22 $79 $0 $37 